# Sentiment Analysis - LSTM Model

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
import os
import sys

src_path = os.path.abspath('../src')
sys.path.append(src_path)

from preprocessing.data_utils import DataUtils
from models import train_lstm

In [ ]:
models_output_dir = '../outputs/models/lstm'
logs_output_dir = '../outputs/logs/lstm'

In [ ]:
import gensim

word2vec_path = "GoogleNews-vectors-negative300.bin"
word2vec = gensim.models.KeyedVectors.load_word2vec_format(word2vec_path, binary=True)

In [ ]:
# preprocess and tokenize data
def preprocess_data(df, max_length=128, vocab_size=10000):
    # initialize tokenizer and lemmatizer
    tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
    lemmatizer = WordNetLemmatizer()

    # remove missing or empty values
    df = df.dropna(subset=['Review']).copy()
    df = df[df['Review'].str.strip() != '']

    # extract texts and labels
    texts = df['Review'].astype(str).tolist()
    labels = df['Polarity'].values

    # apply lemmatization
    texts = [" ".join([lemmatizer.lemmatize(word) for word in text.split()]) for text in texts]

    # tokenize and pad sequences
    tokenizer.fit_on_texts(texts)
    sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

    return padded_sequences, np.array(labels), tokenizer

In [ ]:
# preprocess and tokenize data using existing tokenizer
def tokenize_with_existing_tokenizer(df, tokenizer, max_length):
    texts = df['Review'].astype(str).tolist()
    sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
    labels = df['Polarity'].values
    return padded_sequences, labels

In [ ]:
def attention_layer(inputs):
    attention_weights = Dense(1, activation="tanh")(inputs)  # compute attention weights
    attention_weights = Softmax(axis=1)(attention_weights)  # normalize weights to [0,1]
    context_vector = Multiply()([inputs, attention_weights])  # multiply input by attention weights
    context_vector = Lambda(lambda x: tf.reduce_sum(x, axis=1))(context_vector)  # sum over the sequence
    return context_vector

In [ ]:
def build_attention_lstm_model(vocab_size, embedding_dim, max_length, embedding_matrix, learning_rate=0.001):
    # define inputs
    inputs = Input(shape=(max_length,))
    
    # embedding layer
    embedding = Embedding(input_dim=vocab_size, output_dim=embedding_dim, 
                          input_length=max_length, weights=[embedding_matrix], 
                          trainable=False)(inputs)

    # BiLSTM layer
    lstm_output = Bidirectional(LSTM(128, return_sequences=True))(embedding)  

    # attention layer
    attention_output = attention_layer(lstm_output)

    # fully connected layers
    dropout1 = Dropout(0.5)(attention_output)
    dense1 = Dense(64, activation='relu')(dropout1)
    dropout2 = Dropout(0.5)(dense1)
    outputs = Dense(1, activation='sigmoid')(dropout2)

    # define the model
    model = Model(inputs=inputs, outputs=outputs)

    # compile the model
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), 
                  loss='binary_crossentropy', metrics=['accuracy'])

    return model

# 1. LSTM Model - Amazon Books Reviews

## 1.1. Loading Data

In [ ]:
# load the dataset
amazon_file = '../data/preprocessed/amazon.csv'
amazon_df = pd.read_csv(amazon_file, engine='python', encoding="ISO-8859-1")

In [ ]:
# preview the data
amazon_df.head()

,Polarity,Review
0,0,As a Man Stinketh
1,1,MY FAVORITE!!
2,1,Epic novel
3,1,Very good source for Web Applications Development
4,1,Great fantasy


In [ ]:
train_val_amazon_df, test_amazon_df = train_test_split(
    amazon_df, 
    test_size=0.15, 
    random_state=42, 
    stratify=amazon_df['Polarity']
)

In [ ]:
train_amazon_df, val_amazon_df = train_test_split(
    train_val_amazon_df, 
    test_size=0.1765, # 15% of the original data
    random_state=42, 
    stratify=train_val_amazon_df['Polarity']
)

In [ ]:
DataUtils.count_reviews_by_polarity(train_amazon_df)

In [ ]:
DataUtils.count_reviews_by_polarity(val_amazon_df)

In [ ]:
DataUtils.count_reviews_by_polarity(test_amazon_df)

In [ ]:
DataUtils.plot_token_distribution(train_amazon_df, "Amazon Train Set")

In [ ]:
DataUtils.plot_token_distribution(val_amazon_df, "Amazon Validation Set")

In [ ]:
DataUtils.plot_token_distribution(test_amazon_df, "Amazon Test Set")

## 1.2. Data Preprocessing

In [ ]:
# set preprocessing parameters
vocab_size = 10000  # maximum number of words in the tokenizer
max_length = 128  # maximum length of input sequences

In [ ]:
# preprocess train data and fit tokenizer
X_train, y_train, tokenizer = preprocess_and_prepare(train_amazon_df, max_length=max_length, vocab_size=vocab_size)

In [ ]:
# preprocess val data 
X_val, y_val = tokenize_with_existing_tokenizer(val_amazon_df, tokenizer, max_length)

In [ ]:
# preprocess test data
X_test, y_test = tokenize_with_existing_tokenizer(test_amazon_df, tokenizer, max_length)

## 1.3. Model Training

In [ ]:
# set model parameters
batch_size = 32
epochs = 10
learning_rate = 0.001

In [ ]:
# set embedding parameters
embedding_dim = 300 #word2vec size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        if word in word2vec:
            embedding_matrix[i] = word2vec[word]

In [ ]:
# define model architecture
model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  
])

In [ ]:
model_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
# define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
# train the model
history_lstm = model_lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping]
)

In [ ]:
# plot accuracy on training and validation data
plt.plot(history_lstm.history['accuracy'], label='Training Accuracy')
plt.plot(history_lstm.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
# plot loss on training and validation data
plt.plot(history_lstm.history['loss'], label='Training Loss')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
y_val_pred = (model_lstm.predict(X_val) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_val_pred))

## 1.4. Searching for Best Model Parameters

### 1.4.1. BiLSTM

In [ ]:

model_bilstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, 
              input_length=max_length, weights=[embedding_matrix], 
              trainable=False),
    Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=l2(0.01))),
    Bidirectional(LSTM(64, kernel_regularizer=l2(0.01))),
    Dropout(0.5),
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  
])

In [ ]:
model_bilstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
# train the model
history_bilstm = model_bilstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping]
)

In [ ]:
# plot accuracy on training and validation data
plt.plot(history_bilstm.history['accuracy'], label='Training Accuracy')
plt.plot(history_bilstm.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
# plot loss on training and validation data
plt.plot(history_bilstm.history['loss'], label='Training Loss')
plt.plot(history_bilstm.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
y_val_pred = (model_bilstm.predict(X_val) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_val_pred))

### 1.4.2. BiLSTM + Attention

In [ ]:
model_attention = build_attention_lstm_model(vocab_size=vocab_size, embedding_dim=embedding_dim, max_length=max_length, embedding_matrix=embedding_matrix, learning_rate=learning_rate)

In [ ]:
# train the model
history_attention = model_attention.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping]
)

In [ ]:
# plot accuracy on training and validation data
plt.plot(history_attention.history['accuracy'], label='Training Accuracy')
plt.plot(history_attention.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
# plot loss on training and validation data
plt.plot(history_attention.history['loss'], label='Training Loss')
plt.plot(history_attention.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
y_val_pred = (model_attention.predict(X_val) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_val_pred))

## 1.5. Model Evaluation on Test Data

In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_amazon_df, max_length=max_length)

## 1.6. Model Evaluation on Augmented Data

In [ ]:
# load the augmented test sets
test_amazon_char_swap_file = '../data/preprocessed/amazon_test_char_swap.csv'
test_amazon_char_swap_df = pd.read_csv(test_amazon_char_swap_file, engine='python', encoding="ISO-8859-1")

test_amazon_embedding_file = '../data/preprocessed/amazon_test_embedding.csv'
test_amazon_embedding_df = pd.read_csv(test_amazon_embedding_file, engine='python', encoding="ISO-8859-1")

test_amazon_eda_file = '../data/preprocessed/amazon_test_eda.csv'
test_amazon_eda_df = pd.read_csv(test_amazon_eda_file, engine='python', encoding="ISO-8859-1")

In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_amazon_char_swap_df, max_length=max_length)

In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_amazon_embedding_df, max_length=max_length)


In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_amazon_eda_df, max_length=max_length)


## 1.7. Save Model

In [ ]:
model_path = os.path.join(models_output_dir, "amazon_lstm_model.keras")
model.save(model_path)

# 2. LSTM Model - Sentiment140

## 2.1. Loading Data

In [ ]:
# load the dataset
sentiment140_file = '../data/preprocessed/sentiment140.csv'
sentiment140_df = pd.read_csv(sentiment140_file, engine='python', encoding="ISO-8859-1")

In [ ]:
# preview the data
sentiment140_df.head()

,Polarity,Review
0,0,As a Man Stinketh
1,1,MY FAVORITE!!
2,1,Epic novel
3,1,Very good source for Web Applications Development
4,1,Great fantasy


In [ ]:
train_val_sentiment140_df, test_sentiment140_df = train_test_split(
    sentiment140_df, 
    test_size=0.15, 
    random_state=42, 
    stratify=sentiment140_df['Polarity']
)

In [ ]:
train_sentiment140_df, val_sentiment140_df = train_test_split(
    train_val_sentiment140_df, 
    test_size=0.1765, # 15% of the original data
    random_state=42, 
    stratify=train_val_sentiment140_df['Polarity']
)

In [ ]:
DataUtils.count_reviews_by_polarity(train_sentiment140_df)

In [ ]:
DataUtils.count_reviews_by_polarity(val_sentiment140_df)

In [ ]:
DataUtils.count_reviews_by_polarity(test_sentiment140_df)

In [ ]:
DataUtils.plot_token_distribution(train_sentiment140_df, "Sentiment140 Train Set")

In [ ]:
DataUtils.plot_token_distribution(val_sentiment140_df, "Sentiment140 Validation Set")

In [ ]:
DataUtils.plot_token_distribution(test_sentiment140_df, "Sentiment140 Test Set")

## 2.2. Data Preprocessing

In [ ]:
# set preprocessing parameters
vocab_size = 10000  # maximum number of words in the tokenizer
max_length = 128  # maximum length of input sequences

In [ ]:
# preprocess train data and fit tokenizer
X_train, y_train, tokenizer = preprocess_and_prepare(train_sentiment140_df, max_length=max_length, vocab_size=vocab_size)

In [ ]:
# preprocess val data 
X_val, y_val = tokenize_with_existing_tokenizer(val_sentiment140_df, tokenizer, max_length)

In [ ]:
# preprocess test data
X_test, y_test = tokenize_with_existing_tokenizer(test_sentiment140_df, tokenizer, max_length)

## 2.3. Model Training

In [ ]:
# set model parameters
batch_size = 32
epochs = 10
learning_rate = 0.001

In [ ]:
# set embedding parameters
embedding_dim = 300 #word2vec size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        if word in word2vec:
            embedding_matrix[i] = word2vec[word]

In [ ]:
# define model architecture
model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  
])

In [ ]:
model_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
# define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
# train the model
history_lstm = model_lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping]
)

In [ ]:
# plot accuracy on training and validation data
plt.plot(history_lstm.history['accuracy'], label='Training Accuracy')
plt.plot(history_lstm.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
# plot loss on training and validation data
plt.plot(history_lstm.history['loss'], label='Training Loss')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
y_val_pred = (model_lstm.predict(X_val) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_val_pred))

## 2.4. Searching for Best Model Parameters

### 2.4.1. BiLSTM

In [ ]:

model_bilstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, 
              input_length=max_length, weights=[embedding_matrix], 
              trainable=False),
    Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=l2(0.01))),
    Bidirectional(LSTM(64, kernel_regularizer=l2(0.01))),
    Dropout(0.5),
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  
])

In [ ]:
model_bilstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
# train the model
history_bilstm = model_bilstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping]
)

In [ ]:
# plot accuracy on training and validation data
plt.plot(history_bilstm.history['accuracy'], label='Training Accuracy')
plt.plot(history_bilstm.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
# plot loss on training and validation data
plt.plot(history_bilstm.history['loss'], label='Training Loss')
plt.plot(history_bilstm.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
y_val_pred = (model_bilstm.predict(X_val) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_val_pred))

### 2.4.2. BiLSTM + Attention

In [ ]:
model_attention = build_attention_lstm_model(vocab_size=vocab_size, embedding_dim=embedding_dim, max_length=max_length, embedding_matrix=embedding_matrix, learning_rate=learning_rate)

In [ ]:
# train the model
history_attention = model_attention.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping]
)

In [ ]:
# plot accuracy on training and validation data
plt.plot(history_attention.history['accuracy'], label='Training Accuracy')
plt.plot(history_attention.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
# plot loss on training and validation data
plt.plot(history_attention.history['loss'], label='Training Loss')
plt.plot(history_attention.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
y_val_pred = (model_attention.predict(X_val) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_val_pred))

## 2.5. Model Evaluation on Test Data

In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_sentiment140_df, max_length=max_length)

## 2.6. Model Evaluation on Augmented Data

In [ ]:
# load the augmented test sets
test_sentiment140_char_swap_file = '../data/preprocessed/sentiment140_test_char_swap.csv'
test_sentiment140_char_swap_df = pd.read_csv(test_sentiment140_char_swap_file, engine='python', encoding="ISO-8859-1")

test_sentiment140_embedding_file = '../data/preprocessed/sentiment140_test_embedding.csv'
test_sentiment140_embedding_df = pd.read_csv(test_sentiment140_embedding_file, engine='python', encoding="ISO-8859-1")

test_sentiment140_eda_file = '../data/preprocessed/sentiment140_test_eda.csv'
test_sentiment140_eda_df = pd.read_csv(test_sentiment140_eda_file, engine='python', encoding="ISO-8859-1")

In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_sentiment140_char_swap_df, max_length=max_length)

In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_sentiment140_embedding_df, max_length=max_length)


In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_sentiment140_eda_df, max_length=max_length)


## 2.7. Save Model

In [ ]:
model_path = os.path.join(models_output_dir, "sentiment140_lstm_model.keras")
model.save(model_path)

# 3. LSTM Model - Yelp Reviews

## 3.1. Loading Data

In [ ]:
# load the dataset
yelp_file = '../data/preprocessed/yelp.csv'
yelp_df = pd.read_csv(yelp_file, engine='python', encoding="ISO-8859-1")

In [ ]:
# preview the data
yelp_df.head()

,Polarity,Review
0,0,As a Man Stinketh
1,1,MY FAVORITE!!
2,1,Epic novel
3,1,Very good source for Web Applications Development
4,1,Great fantasy


In [ ]:
train_val_yelp_df, test_yelp_df = train_test_split(
    yelp_df, 
    test_size=0.15, 
    random_state=42, 
    stratify=yelp_df['Polarity']
)

In [ ]:
train_yelp_df, val_yelp_df = train_test_split(
    train_val_yelp_df, 
    test_size=0.1765, # 15% of the original data
    random_state=42, 
    stratify=train_val_yelp_df['Polarity']
)

In [ ]:
DataUtils.count_reviews_by_polarity(train_yelp_df)

In [ ]:
DataUtils.count_reviews_by_polarity(val_yelp_df)

In [ ]:
DataUtils.count_reviews_by_polarity(test_yelp_df)

In [ ]:
DataUtils.plot_token_distribution(train_yelp_df, "Yelp Train Set")

In [ ]:
DataUtils.plot_token_distribution(val_yelp_df, "Yelp Validation Set")

In [ ]:
DataUtils.plot_token_distribution(test_yelp_df, "Yelp Test Set")

## 3.2. Data Preprocessing

In [ ]:
# set preprocessing parameters
vocab_size = 10000  # maximum number of words in the tokenizer
max_length = 256  # maximum length of input sequences

In [ ]:
# preprocess train data and fit tokenizer
X_train, y_train, tokenizer = preprocess_and_prepare(train_yelp_df, max_length=max_length, vocab_size=vocab_size)

In [ ]:
# preprocess val data 
X_val, y_val = tokenize_with_existing_tokenizer(val_yelp_df, tokenizer, max_length)

In [ ]:
# preprocess test data
X_test, y_test = tokenize_with_existing_tokenizer(test_yelp_df, tokenizer, max_length)

## 3.3. Model Training

In [ ]:
# set model parameters
batch_size = 32
epochs = 10
learning_rate = 0.001

In [ ]:
# set embedding parameters
embedding_dim = 300 #word2vec size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i < vocab_size:
        if word in word2vec:
            embedding_matrix[i] = word2vec[word]

In [ ]:
# define model architecture
model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  
])

In [ ]:
model_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
# define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
# train the model
history_lstm = model_lstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping]
)

In [ ]:
# plot accuracy on training and validation data
plt.plot(history_lstm.history['accuracy'], label='Training Accuracy')
plt.plot(history_lstm.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
# plot loss on training and validation data
plt.plot(history_lstm.history['loss'], label='Training Loss')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
y_val_pred = (model_lstm.predict(X_val) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_val_pred))

## 3.4. Searching for Best Model Parameters

### 3.4.1. BiLSTM

In [ ]:

model_bilstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, 
              input_length=max_length, weights=[embedding_matrix], 
              trainable=False),
    Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=l2(0.01))),
    Bidirectional(LSTM(64, kernel_regularizer=l2(0.01))),
    Dropout(0.5),
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  
])

In [ ]:
model_bilstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [ ]:
# train the model
history_bilstm = model_bilstm.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping]
)

In [ ]:
# plot accuracy on training and validation data
plt.plot(history_bilstm.history['accuracy'], label='Training Accuracy')
plt.plot(history_bilstm.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
# plot loss on training and validation data
plt.plot(history_bilstm.history['loss'], label='Training Loss')
plt.plot(history_bilstm.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
y_val_pred = (model_bilstm.predict(X_val) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_val_pred))

### 3.4.2. BiLSTM + Attention

In [ ]:
model_attention = build_attention_lstm_model(vocab_size=vocab_size, embedding_dim=embedding_dim, max_length=max_length, embedding_matrix=embedding_matrix, learning_rate=learning_rate)

In [ ]:
# train the model
history_attention = model_attention.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[early_stopping]
)

In [ ]:
# plot accuracy on training and validation data
plt.plot(history_attention.history['accuracy'], label='Training Accuracy')
plt.plot(history_attention.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

In [ ]:
# plot loss on training and validation data
plt.plot(history_attention.history['loss'], label='Training Loss')
plt.plot(history_attention.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

In [ ]:
# evaluate the model on the validation set
print("\nValidation Results:")
y_val_pred = (model_attention.predict(X_val) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_val_pred))

## 3.5. Model Evaluation on Test Data

In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_yelp_df, max_length=max_length)

## 3.6. Model Evaluation on Augmented Data

In [ ]:
# load the augmented test sets
test_yelp_char_swap_file = '../data/preprocessed/yelp_test_char_swap.csv'
test_yelp_char_swap_df = pd.read_csv(test_yelp_char_swap_file, engine='python', encoding="ISO-8859-1")

test_yelp_transform_file = '../data/preprocessed/yelp_test_transform.csv'
test_yelp_transform_df = pd.read_csv(test_yelp_transform_file, engine='python', encoding="ISO-8859-1")

test_yelp_combined_file = '../data/preprocessed/yelp_test_combined.csv'
test_yelp_combined_df = pd.read_csv(test_yelp_combined_file, engine='python', encoding="ISO-8859-1")

In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_yelp_char_swap_df, max_length=max_length)

In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_yelp_transform_df, max_length=max_length)


In [ ]:
train_lstm.evaluate_model(model=model, tokenizer=tokenizer, test=test_yelp_combined_df, max_length=max_length)


## 3.7. Save Model

In [ ]:
model_path = os.path.join(models_output_dir, "yelp_lstm_model.keras")
model.save(model_path)